In [4]:
import pandas as pd
import folium
from folium import plugins
from folium.plugins import MarkerCluster
import branca.colormap as cm


In [40]:
def plot_energy_flows(results_df, supply_df, demand_df, save_path=None, draw_lines=True):
    supply_locations = (supply_df[['Supply Site', 'Latitude', 'Longitude']]
                        .drop_duplicates()
    )
    demand_locations = (demand_df[['Demand Site', 'Latitude', 'Longitude']]
                        .drop_duplicates()
    )
    supply_location_dict = supply_locations.set_index('Supply Site')[['Latitude', 'Longitude']].to_dict('index')
    demand_location_dict = demand_locations.set_index('Demand Site')[['Latitude', 'Longitude']].to_dict('index')
    print(supply_location_dict.keys())
    # Merge energy data with location coordinates
    results_df[['Supply Latitude']] = results_df[['Supply Site']].map(lambda s: supply_location_dict[s]['Latitude'])
    results_df[['Supply Longitude']] = results_df[['Supply Site']].map(lambda s: supply_location_dict[s]['Longitude'])
    results_df[['Demand Latitude']] = results_df[['Demand Site']].map(lambda s: demand_location_dict[s]['Latitude'])
    results_df[['Demand Longitude']] = results_df[['Demand Site']].map(lambda s: demand_location_dict[s]['Longitude'])

    # Central point for the map
    uk_centre = (53.9819, -2.5421)

    # Initialise folium map
    m = folium.Map(location=uk_centre, zoom_start=6, tiles="OpenStreetMap"
)

    supply_cluster = MarkerCluster(name='Supply Sites').add_to(m)
    for _, row in supply_locations.iterrows():
        folium.Marker(
            location=[row['Latitude'], row['Longitude']],
            popup=f"Supply Site: {row['Supply Site']}",
            icon=folium.Icon(color='blue', icon='bolt', prefix='fa')
        ).add_to(m)

    demand_cluster = MarkerCluster(name='Demand Sites').add_to(m)
    for _, row in demand_locations.iterrows():
        folium.Marker(
            location=[row['Latitude'], row['Longitude']],
            popup=f"Demand Site: {row['Demand Site']}",
            icon=folium.Icon(color='red', icon='home')
        ).add_to(m)

    # unique_years = sorted(results_df['Year'].unique())
    # colormap = cm.linear.Set1_09.scale(min(unique_years), max(unique_years))
    aggregated_results = results_df.groupby(['Supply Site', 'Demand Site',
                                             'Supply Latitude', 'Supply Longitude', 
                                             'Demand Latitude', 'Demand Longitude'])['Energy Provided'].sum().reset_index()

    if draw_lines:
        energy_max = aggregated_results['Energy Provided'].max()
        for _, row in aggregated_results.iterrows():
            # line_color = colormap(row['Year'])
            line_width = 1 + 4 * (row['Energy Provided']/energy_max)
            # Create line
            folium.PolyLine(
                locations=[[row['Supply Latitude'], row['Supply Longitude']],
                        [row['Demand Latitude'], row['Demand Longitude']]],
                color='blue',
                weight=line_width,
                opacity=0.6,
            ).add_to(m)

        # colormap.caption ='Year'
        # colormap.add_to(m)

        folium.LayerControl().add_to(m)

    title_html = "test title for now"
    m.get_root().html.add_child(folium.Element(title_html))

    if save_path:
        m.save(save_path)

    
    return m




In [11]:
# # Intro example
supply_df = pd.read_csv("Data/dummy_test/dummy_supply.csv").rename(columns={'Site Name': "Supply Site"})
demand_df = pd.read_csv("Data/dummy_test/dummy_demand.csv").rename(columns={'Local authority': 'Demand Site'})
results_df = pd.read_csv("Outputs/supply_results_baseline_baseline_1.csv").rename(columns={'Energy Provided (MW)': "Energy Provided"})
# # results_df.columns

#Base case 1
# map = plot_energy_flows(results_df, supply_df, demand_df, draw_lines=False, save_path="report/figures/intro_example.html")
results_esl = pd.read_csv("Outputs/supply_results_baseline_baseline_1.csv").rename(columns={'Energy Provided (MW)': "Energy Provided"})
results_lr = pd.read_csv("Outputs/supply_results_LR_baseline_1.csv").rename(columns={'Energy Provided (MW)': "Energy Provided"})
map = plot_energy_flows(results_df, supply_df, demand_df, draw_lines=True, save_path="report/figures/base_case_1_test")
map = plot_energy_flows(results_df, supply_df, demand_df, draw_lines=True, save_path="report/figures/lr_heur_case_1_test")
map

dict_keys(['Glendoe Hydro Scheme', 'Gate Burton - Solar & Energy Storage Park', 'Berwick Bank Offshore Wind Farm'])
dict_keys(['Glendoe Hydro Scheme', 'Gate Burton - Solar & Energy Storage Park', 'Berwick Bank Offshore Wind Farm'])


In [3]:
map = plot_energy_flows(results_df, supply_df, demand_df, draw_lines=False, save_path="report/figures/intro_example.html")
results_eslp = pd.read_csv("Outputs/supply_results_baseline_baseline_2.csv").rename(columns={'Energy Provided (MW)': "Energy Provided"})
results_lr = pd.read_csv("Outputs/supply_results_LR_baseline_2.csv").rename(columns={'Energy Provided (MW)': "Energy Provided"})
map = plot_energy_flows(results_eslp, supply_df, demand_df, draw_lines=True, save_path="report/figures/base_case_2.html")
map = plot_energy_flows(results_lr, supply_df, demand_df, draw_lines=True, save_path="report/figures/lr_heur_case_2.html")
map

NameError: name 'plot_energy_flows' is not defined

In [41]:
import pandas as pd
supply_df = pd.read_csv("Data/test_8_supply.csv")
demand_df = pd.read_csv("Data/test_8_demand.csv").dropna()
results_eslp = pd.read_csv("Outputs/supply_results_baseline_test_8.csv").rename(columns={'Energy Provided (MW)': "Energy Provided"})
results_lr = pd.read_csv("Outputs/supply_results_LR_test_8.csv").rename(columns={'Energy Provided (MW)': "Energy Provided"})
map = plot_energy_flows(results_eslp, supply_df, demand_df, draw_lines=True, save_path="report/figures/test_8_base.html")
map = plot_energy_flows(results_lr, supply_df, demand_df, draw_lines=True, save_path="report/figures/test_8_lr.html")
map


dict_keys(['Glendoe Hydro Scheme', 'Lochaber', 'Berwick Bank Offshore Wind Farm', 'Tummel Bridge Power Station - Hydro Plant', 'Great Glen Scheme', 'Dolgarrog', 'Hornsea 3', 'The East Anglia Array - Norfolk Vanguard', 'Hornsea 4', 'Dogger Bank A & B (was Creyke Beck A & B)'])
dict_keys(['Glendoe Hydro Scheme', 'Lochaber', 'Berwick Bank Offshore Wind Farm', 'Tummel Bridge Power Station - Hydro Plant', 'Great Glen Scheme', 'Dolgarrog', 'Hornsea 3', 'The East Anglia Array - Norfolk Vanguard', 'Hornsea 4', 'Dogger Bank A & B (was Creyke Beck A & B)'])


In [42]:
supply_df = pd.read_csv("Data/test_12_supply.csv")
demand_df = pd.read_csv("Data/test_12_demand.csv").dropna()
results_eslp = pd.read_csv("Outputs/supply_results_baseline_test_12.csv").rename(columns={'Energy Provided (MW)': "Energy Provided"})
results_lr = pd.read_csv("Outputs/supply_results_LR_test_12.csv").rename(columns={'Energy Provided (MW)': "Energy Provided"})
map = plot_energy_flows(results_eslp, supply_df, demand_df, draw_lines=True, save_path="report/figures/test_12_base.html")
map = plot_energy_flows(results_lr, supply_df, demand_df, draw_lines=True, save_path="report/figures/test_12_lr.html")
map


dict_keys(['Glendoe Hydro Scheme', 'Lochaber', 'Berwick Bank Offshore Wind Farm', 'Tummel Bridge Power Station - Hydro Plant', 'Great Glen Scheme', 'Dolgarrog', 'Hornsea 3', 'The East Anglia Array - Norfolk Vanguard', 'Hornsea 4', 'Dogger Bank A & B (was Creyke Beck A & B)', 'West of Orkney Wind Farm', 'Aigas', 'Kilmorack', 'Ceannacroc', 'Invergarry', 'Kinlochleven Hydro Power Station', 'Orrin', 'Mona Offshore Wind Farm', 'Outer Dowsing Offshore Wind Farm', 'Pitlochry', 'Tor Achilty', 'Livishie', 'East Anglia 3 (EA 3)', 'The East Anglia Array - Norfolk Boreas', 'Sofia (Teesside B)', 'Hornsea 2 - Optimus and Breesea', 'Hornsea 1 - Heron & Njord', 'Atlantic Array One - Bristol Channel Zone', 'Dogger Bank C (was Teesside A)', 'Inch Cape', 'Seagreen', 'Cashlie', 'Stronelarig Hydro Scheme', 'Navitus Bay', 'Moray East', 'Cassley', 'Cwm Dyli', 'East Anglia 2 (EA 2)', 'Moray West Offshore Wind Farm Project', 'Triton Knoll\xa0', 'River Pattack Hydro Scheme', 'East Anglia 1 North (EA 4)', 'Striv

In [45]:
supply_df = pd.read_csv("Data/test_14_supply.csv")
demand_df = pd.read_csv("Data/test_14_demand.csv").dropna()
results_eslp = pd.read_csv("Outputs/supply_results_baseline_test_14.csv").rename(columns={'Energy Provided (MW)': "Energy Provided"})
results_lr = pd.read_csv("Outputs/supply_results_LR_test_14.csv").rename(columns={'Energy Provided (MW)': "Energy Provided"})
map = plot_energy_flows(results_eslp, supply_df, demand_df, draw_lines=True, save_path="report/figures/test_14_base.html")
map = plot_energy_flows(results_lr, supply_df, demand_df, draw_lines=True, save_path="report/figures/test_14_lr.html")
map

dict_keys(['Glendoe Hydro Scheme', 'Lochaber', 'Berwick Bank Offshore Wind Farm', 'Tummel Bridge Power Station - Hydro Plant', 'Great Glen Scheme', 'Dolgarrog', 'Hornsea 3', 'The East Anglia Array - Norfolk Vanguard', 'Hornsea 4', 'Dogger Bank A & B (was Creyke Beck A & B)', 'West of Orkney Wind Farm', 'Aigas', 'Kilmorack', 'Ceannacroc', 'Invergarry', 'Kinlochleven Hydro Power Station', 'Orrin', 'Mona Offshore Wind Farm', 'Outer Dowsing Offshore Wind Farm', 'Pitlochry', 'Tor Achilty', 'Livishie', 'East Anglia 3 (EA 3)', 'The East Anglia Array - Norfolk Boreas', 'Sofia (Teesside B)', 'Hornsea 2 - Optimus and Breesea', 'Hornsea 1 - Heron & Njord', 'Atlantic Array One - Bristol Channel Zone', 'Dogger Bank C (was Teesside A)', 'Inch Cape', 'Seagreen', 'Cashlie', 'Stronelarig Hydro Scheme', 'Navitus Bay', 'Moray East', 'Cassley', 'Cwm Dyli', 'East Anglia 2 (EA 2)', 'Moray West Offshore Wind Farm Project', 'Triton Knoll\xa0', 'River Pattack Hydro Scheme', 'East Anglia 1 North (EA 4)', 'Striv